# Chapter 03 — Supervised Learning: Live Examples

**Session 1 | Chapter 3 | Duration: ~15 min**

This notebook visualizes the key concepts of supervised learning:
- Underfitting vs Overfitting
- The Bias-Variance Tradeoff
- Cross-Validation in action

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error

sns.set_theme(style='whitegrid')
np.random.seed(42)
print('Ready!')

## 1. Generating Data

We create a simple dataset with a known true function (a sine wave) + noise.  
This is useful because we *know* the right answer, so we can clearly see what models learn.

In [ ]:
# True underlying function: y = sin(x) + noise
n_samples = 30
X_raw = np.sort(np.random.uniform(0, 2 * np.pi, n_samples))
y = np.sin(X_raw) + np.random.normal(0, 0.2, n_samples)  # true signal + noise

# Reshape for sklearn (needs 2D input)
X = X_raw.reshape(-1, 1)

# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# True function (for plotting)
X_plot = np.linspace(0, 2 * np.pi, 300).reshape(-1, 1)
y_true = np.sin(X_plot)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(X_plot, y_true, '--', color='gray', label='True function (sin)', linewidth=2)
ax.scatter(X_train, y_train, color='#3498db', label='Training data', s=80, zorder=5)
ax.scatter(X_test, y_test, color='#e74c3c', marker='^', label='Test data', s=80, zorder=5)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Our Dataset: True Function + Noise', fontsize=14)
ax.legend()
plt.tight_layout()
plt.show()

## 2. Underfitting, Good Fit, Overfitting

We fit polynomial models of increasing degree.  
- Degree 1 = linear = **underfitting**
- Degree 4 = **good fit**
- Degree 15 = **overfitting** (memorizes training noise)

In [ ]:
degrees = [1, 4, 15]
titles = ['Degree 1 — Underfitting', 'Degree 4 — Good Fit', 'Degree 15 — Overfitting']
colors = ['#e74c3c', '#2ecc71', '#9b59b6']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, degree, title, color in zip(axes, degrees, titles, colors):
    # Create and train the model
    model = Pipeline([
        ('poly', PolynomialFeatures(degree=degree)),
        ('linear', LinearRegression())
    ])
    model.fit(X_train, y_train)

    # Compute errors
    train_rmse = np.sqrt(mean_squared_error(y_train, model.predict(X_train)))
    test_rmse  = np.sqrt(mean_squared_error(y_test,  model.predict(X_test)))

    # Plot
    ax.plot(X_plot, y_true, '--', color='gray', linewidth=1.5, label='True function')
    ax.scatter(X_train, y_train, color='#3498db', s=60, label='Train data', zorder=5)
    ax.scatter(X_test,  y_test,  color='#e74c3c', marker='^', s=60, label='Test data', zorder=5)
    ax.plot(X_plot, model.predict(X_plot), color=color, linewidth=2.5, label='Model')

    ax.set_title(f'{title}\nTrain RMSE: {train_rmse:.3f} | Test RMSE: {test_rmse:.3f}', fontsize=12)
    ax.set_xlabel('x')
    ax.set_ylim(-2, 2)
    ax.legend(fontsize=9)

plt.suptitle('Underfitting → Good Fit → Overfitting', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print('\nObservations:')
print('Degree 1:  High train AND test error → underfitting (model too simple)')
print('Degree 4:  Low train error AND low test error → good generalization')
print('Degree 15: Low train error BUT high test error → overfitting (memorized noise)')

## 3. The Bias-Variance Curve

Let's plot train error and test error across all polynomial degrees.  
This shows the classic U-shape for test error.

In [ ]:
max_degree = 15
train_errors, test_errors = [], []

for d in range(1, max_degree + 1):
    model = Pipeline([
        ('poly', PolynomialFeatures(degree=d)),
        ('linear', LinearRegression())
    ])
    model.fit(X_train, y_train)
    train_errors.append(np.sqrt(mean_squared_error(y_train, model.predict(X_train))))
    test_errors.append(np.sqrt(mean_squared_error(y_test, model.predict(X_test))))

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(range(1, max_degree + 1), train_errors, 'o-', color='#3498db', label='Train RMSE', linewidth=2)
ax.plot(range(1, max_degree + 1), test_errors,  's-', color='#e74c3c', label='Test RMSE',  linewidth=2)

ax.axvline(x=4, color='#2ecc71', linestyle='--', alpha=0.7, label='Sweet spot (degree 4)')
ax.annotate('Underfitting\n(high bias)', xy=(1, test_errors[0]), xytext=(2, 0.7),
            arrowprops=dict(arrowstyle='->', color='gray'), fontsize=11, color='gray')
ax.annotate('Overfitting\n(high variance)', xy=(12, test_errors[11]), xytext=(10, 0.9),
            arrowprops=dict(arrowstyle='->', color='gray'), fontsize=11, color='gray')

ax.set_xlabel('Polynomial Degree (Model Complexity)', fontsize=13)
ax.set_ylabel('RMSE', fontsize=13)
ax.set_title('Bias-Variance Tradeoff: Train vs Test Error', fontsize=14)
ax.legend(fontsize=12)
ax.set_ylim(0, 1.2)
plt.tight_layout()
plt.show()

## 4. Cross-Validation: Getting Robust Estimates

In [ ]:
# Compare a single train/test split estimate vs. 5-fold cross-validation
model_d4 = Pipeline([
    ('poly', PolynomialFeatures(degree=4)),
    ('linear', LinearRegression())
])

# Single split
model_d4.fit(X_train, y_train)
single_test_rmse = np.sqrt(mean_squared_error(y_test, model_d4.predict(X_test)))
print(f'Single test split RMSE:   {single_test_rmse:.4f}')

# 5-fold cross-validation (on the full dataset)
cv_scores = cross_val_score(
    model_d4, X, y,
    cv=5,
    scoring='neg_root_mean_squared_error'
)
cv_rmse = -cv_scores  # sklearn returns negative values

print(f'\n5-Fold CV RMSE scores:    {cv_rmse.round(4)}')
print(f'CV Mean RMSE:             {cv_rmse.mean():.4f}')
print(f'CV Std RMSE:              {cv_rmse.std():.4f}')
print('\nThe ± tells us how stable the estimate is.')
print('With cross-validation, we use ALL data for both training and evaluation.')

In [ ]:
# Visualize cross-validation fold scores for different degrees
cv_means, cv_stds = [], []

for d in range(1, max_degree + 1):
    m = Pipeline([('poly', PolynomialFeatures(degree=d)), ('linear', LinearRegression())])
    scores = -cross_val_score(m, X, y, cv=5, scoring='neg_root_mean_squared_error')
    cv_means.append(scores.mean())
    cv_stds.append(scores.std())

cv_means = np.array(cv_means)
cv_stds = np.array(cv_stds)
best_degree = np.argmin(cv_means) + 1

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(range(1, max_degree + 1), cv_means, 'o-', color='#9b59b6', linewidth=2, label='CV Mean RMSE')
ax.fill_between(range(1, max_degree + 1),
                cv_means - cv_stds, cv_means + cv_stds,
                alpha=0.2, color='#9b59b6', label='±1 std')
ax.axvline(x=best_degree, color='#2ecc71', linestyle='--', label=f'Best degree = {best_degree}')

ax.set_xlabel('Polynomial Degree', fontsize=13)
ax.set_ylabel('Cross-Validated RMSE', fontsize=13)
ax.set_title('Model Selection with Cross-Validation', fontsize=14)
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

print(f'\nCross-validation selected degree = {best_degree} as the best model.')
print('This matches what we expected from the bias-variance plot!')

## Summary

| Concept | What we saw |
|---------|------------|
| Underfitting | Degree 1 — couldn't capture the sine shape |
| Good fit | Degree 4 — low train AND test error |
| Overfitting | Degree 15 — low train error, high test error |
| Bias-Variance | Classic U-curve for test error vs. complexity |
| Cross-validation | Automatically selects the best model degree |

**Next: Chapter 04 — Let's meet the actual regression algorithms!**